In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import scipy.sparse as sp
import doubletdetection as dd

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/ouptuts/h5ad_files/inprogress.h5ad')

In [ ]:
adata

In [ ]:
# choose parameters here
DD_PARAMS = dict(
    n_iters=50,
    clustering_algorithm="louvain",  # or 'leiden'
    standard_scaling=True,
    pseudocount=0.1,
    n_jobs=8,
    random_state=0,
)
P_THRESH   = 1e-16   # very stringent; adjust as needed
VOTER_THRS = 0.5

# storage
dd_score  = pd.Series(index=adata.obs_names, dtype="float64")
dd_pred   = pd.Series(index=adata.obs_names, dtype="boolean")

for b in adata.obs["batch"].unique():
    idx = adata.obs["batch"] == b
    ad_b = adata[idx].copy()

    # get raw counts
    X = ad_b.layers["counts"]
    if sp.issparse(X):
        X = X.tocsr()  # DoubletDetection can take sparse CSR or dense
    # fit/predict
    clf = dd.BoostClassifier(**DD_PARAMS)
    doublets = clf.fit(X).predict(p_thresh=P_THRESH, voter_thresh=VOTER_THRS)

    # collect
    dd_score.loc[ad_b.obs_names] = clf.doublet_score()
    dd_pred.loc[ad_b.obs_names]  = pd.Series(doublets, index=ad_b.obs_names).astype("boolean")

# write to obs
adata.obs["dd_score"] = dd_score
adata.obs["dd_is_doublet"] = dd_pred
adata.uns["doubletdetection_params"] = {**DD_PARAMS, "p_thresh": P_THRESH, "voter_thresh": VOTER_THRS}
print("DoubletDetection done:", int(adata.obs["dd_is_doublet"].sum()), "doublets flagged.")

In [ ]:
# adata.obs

In [ ]:
# %matplotlib inline

In [ ]:
# adata.obs["dd_is_doublet_num"] = adata.obs["dd_is_doublet"].astype(int)

In [ ]:
# sc.pl.umap(
#     adata,
#     color = ['dd_is_doublet_num'],
#     size = 10
# )

In [ ]:
# --- Scrublet ---
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scrublet as scr

# Parameters
SCR_PARAMS = dict(
    expected_doublet_rate=0.05,  # adjust if needed
    sim_doublet_ratio=2.0,
    random_state=0,
)
N_PRIN_COMPS = 30  # moved to scrub_doublets()

scr_score = pd.Series(index=adata.obs_names, dtype="float64")
scr_pred  = pd.Series(index=adata.obs_names, dtype="boolean")
scr_thrs  = pd.Series(index=adata.obs_names, dtype="float64")

for b in adata.obs["batch"].unique():
    idx = adata.obs["batch"] == b
    ad_b = adata[idx].copy()

    # raw counts
    X = ad_b.layers["counts"]
    if sp.issparse(X):
        X = X.tocsr()
    if sp.issparse(X):
        X = X.toarray()  # Scrublet needs dense

    scrub = scr.Scrublet(X, **SCR_PARAMS)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(
        min_counts=2,
        min_cells=3,
        use_approx_neighbors=True,
        n_prin_comps=N_PRIN_COMPS,  # <-- pass here
    )

    # store results
    scr_score.loc[ad_b.obs_names] = doublet_scores
    scr_pred.loc[ad_b.obs_names]  = pd.Series(predicted_doublets, index=ad_b.obs_names).astype("boolean")
    scr_thrs.loc[ad_b.obs_names]  = float(scrub.threshold_)

adata.obs["scrublet_score"] = scr_score
adata.obs["scrublet_is_doublet"] = scr_pred
adata.obs["scrublet_threshold_used"] = scr_thrs
adata.uns["scrublet_params"] = {**SCR_PARAMS, "n_prin_comps": N_PRIN_COMPS}

print("Scrublet done:", int(adata.obs["scrublet_is_doublet"].sum()), "doublets flagged.")


In [ ]:
adata.write_h5ad(f'{ACC_DATA_ROOT}/ouptuts/h5ad_files/inprogress_dd_scr.h5ad', compression = 'lzf')

In [ ]:
# adata.obs

In [ ]:
# adata.obs["scrublet_is_doublet_num"] = adata.obs["scrublet_is_doublet"].astype(int)

In [ ]:
# adata.obs

In [ ]:
# sc.pl.umap( 
#     adata,
#     color = ['scrublet_is_doublet_num'],
#     size = 10
# )

In [ ]:
# --- SOLO (scvi-tools) ---
import scvi
import pandas as pd

# set up scvi with raw counts layer and batch key
scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="batch")

# scVI model (tweak as you like)
scvi_model = scvi.model.SCVI(
    adata,
    n_layers=2,
    n_latent=30,
    gene_likelihood="nb",   # NB for raw counts
)
scvi_model.train(max_epochs=100, check_val_every_n_epoch=10, early_stopping=True, plan_kwargs={"lr": 1e-3})

# build & train SOLO on top of the trained scVI model
from scvi.external import SOLO
solo = SOLO.from_scvi_model(scvi_model)
solo.train(max_epochs=100, early_stopping=True)

# predict; column names can vary by scvi-tools version, so handle robustly
pred_df = solo.predict(adata)  # pandas DataFrame indexed by obs_names

# Try to find score & label columns across versions
score_col_candidates = [c for c in pred_df.columns if "prob" in c.lower() or "score" in c.lower()]
pred_col_candidates  = [c for c in pred_df.columns if "pred" in c.lower() or "doublet" in c.lower()]

if not score_col_candidates:
    raise RuntimeError(f"SOLO predict() did not return a probability/score column. Got: {pred_df.columns.tolist()}")
if not pred_col_candidates:
    raise RuntimeError(f"SOLO predict() did not return a predicted label column. Got: {pred_df.columns.tolist()}")

score_col = score_col_candidates[0]
pred_col  = pred_col_candidates[0]

adata.obs["solo_score"]       = pred_df[score_col].astype(float)
adata.obs["solo_is_doublet"]  = pred_df[pred_col].astype(str).str.contains("doublet", case=False) \
                                if pred_df[pred_col].dtype == "object" else pred_df[pred_col].astype(bool)

adata.uns["solo_info"] = {
    "score_col": score_col,
    "pred_col": pred_col,
    "scvi_n_latent": 30,
    "scvi_n_layers": 2,
    "gene_likelihood": "nb",
}
print("SOLO done:", int(adata.obs["solo_is_doublet"].sum()), "doublets flagged.")
